# The complete model lifecycle: Kedro + MLflow + JupyterLab

Start with `workshop.md` in this directory for the click-by-click instructions.
This notebook is a complete worked example. Read the explanation, click the code cell,
and press **Shift+Enter**. A number such as `[1]` means completion; `[*]` means work is
in progress. Wait for a number before moving on.

**Business goal:** batch valuation of 250 synthetic properties in PLN.
**Before training**, agree on RMSE ≤ PLN 50,000 and RMSE ≤ 70% of the baseline error.
These are teaching thresholds, not real market requirements. Choose candidates on
validation data and check test data at the release gate. Do not tune the grid after
looking at the test results.

Kedro runs the graph and handles datasets; MLflow records experiments and model versions;
JupyterLab explores and controls the same source code. A local batch job is our deployment.
There is no prediction API or scheduler. Kedro-Viz is an optional extra.

Every fresh execution creates its own directory and databases in `.runtime/trening/en/`.
The existing `data/` directory and notebooks remain a separate demonstration.

## 0. Create an isolated exercise workspace

This cell locates the project, creates an execution directory, and configures local
database connections **before** loading Kedro. Keep the printed `WORKSPACE` path:
reports and release manifests are stored there. Paste the printed MLflow command
into a terminal later.

In [ ]:
# ruff: noqa: E402
LANGUAGE = "en"

import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists() and (path / "src/kedro_lasso").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from trening.lifecycle import (
    create_workspace,
    future_frame,
    monitor_inputs,
    quality_gate,
    validate_workspace_connections,
)

WORKSPACE = create_workspace(PROJECT_ROOT, LANGUAGE)
from kedro_lasso.commands import prepare_runtime, seed
from kedro_lasso.config import load_config

prepare_runtime()
config = load_config()
validate_workspace_connections(config, WORKSPACE)
seed()
tracking_uri = config["globals"]["tracking_uri"]
os.environ["MLFLOW_REGISTRY_URI"] = tracking_uri
policy = {"max_rmse_pln": 50_000, "max_baseline_ratio": 0.70}

print("Python:", sys.executable)
print("WORKSPACE:", WORKSPACE)
print("Release policy:", policy)
print("MLflow terminal command (run from the project root):")
print(
    f'uv run python -m mlflow server --backend-store-uri "{tracking_uri}" '
    f'--artifacts-destination "{WORKSPACE.as_posix()}/mlflow/artifacts" '
    "--host 127.0.0.1 --port 5001 --workers 1"
)

## 1. Connect JupyterLab to Kedro

`%load_ext` loads the extension and `%reload_kedro` refreshes the project. The extension
provides `catalog`, `context`, `session`, and `pipelines`. Reload after changing YAML or
source code. Create a new `KedroSession` for each pipeline execution.

In [ ]:
get_ipython().run_line_magic("load_ext", "kedro.ipython")
get_ipython().run_line_magic("reload_kedro", f'"{PROJECT_ROOT}"')
catalog = get_ipython().user_ns["catalog"]
context = get_ipython().user_ns["context"]
pipelines = get_ipython().user_ns["pipelines"]
print("Pipelines:", list(pipelines))
print("Catalog datasets:", list(catalog.keys()))
print("Source:", config["credentials"]["source_db"]["con"])
print("Warehouse:", config["credentials"]["warehouse_db"]["con"])

## 2. Explore and validate the data

The SQLite connection and SQL query live in `conf/base/catalog.yml` and `credentials.yml`.
`catalog.load()` uses the same configuration as batch execution. `price_pln` is the target,
`property_id` is an identifier, and the other configured columns are features. Neither
the ID nor target may be a feature. The model's imputer handles missing feature values.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from kedro_lasso.domain import regression_metrics, validate_frame
from kedro_lasso.training import split_training

raw = catalog.load("raw_training")
raw_scoring = catalog.load("raw_scoring")
schema = context.params["data"]
features = schema["features"]
clean = validate_frame(raw, schema, training=True)
display(raw.head())
display(raw[features + [schema["target"]]].describe().T)
display(raw.isna().mean().rename("missing_rate").to_frame())
assert len(raw) == 1500 and len(raw_scoring) == 250

raw.plot.scatter(x="area_m2", y="price_pln", alpha=0.25, figsize=(8, 4))
plt.ylabel("Price [PLN]")
plt.title("Synthetic properties / Nieruchomości syntetyczne")
plt.show()

### A controlled failure: duplicate IDs

This cell deliberately creates invalid data in memory. Expect `EXPECTED VALIDATION ERROR`,
with no database write. Catching the exception lets us continue.
**Exercise A:** in a new cell, remove one feature and inspect the `Missing columns` message.

In [ ]:
broken = pd.concat([raw, raw.iloc[:1]], ignore_index=True)
try:
    validate_frame(broken, schema, training=True)
except ValueError as error:
    print("EXPECTED VALIDATION ERROR:", error)
else:
    raise AssertionError("Duplicate IDs should have been rejected.")

### Disjoint splits and architecture

Call the same function as the Kedro node instead of copying training logic into the notebook.
Train fits parameters, validation selects `alpha`, and test evaluates that fixed choice.
Each candidate fits its imputer and scaler on its training data only.
**Exercise B:** double-click `domain.py`, `training.py`, and `pipeline_registry.py` in the
file browser. Locate model construction, MLflow logging, and pipeline composition.

In [ ]:
splits = split_training(clean, context.params["split"])
ids = {key: set(frame[schema["id_column"]]) for key, frame in splits.items()}
assert not (
    ids["train"] & ids["validation"]
    or ids["train"] & ids["test"]
    or ids["validation"] & ids["test"]
)
display(pd.Series({key: len(frame) for key, frame in splits.items()}, name="rows"))
print(pipelines["training"].describe())

## 3. Train through Kedro and track in MLflow

Run only `training`: validation → split → experiments → refit → report.
No predictions are published yet. `runtime_params` changes the experiment and run names
without editing YAML. Keep the Kedro session ID, which is also the saved bundle version.
A run is one recorded MLflow execution; nested runs correspond to `alpha` values.

In [ ]:
from kedro.framework.session import KedroSession
from kedro.io import Version
from kedro_datasets.pickle import PickleDataset

EXPERIMENT = "workshop-lifecycle"
REGISTERED_NAME = "workshop-housing-lasso"


def load_bundle(version):
    return PickleDataset(
        filepath=(WORKSPACE / "models" / "lasso.pkl").as_posix(),
        version=Version(version, None),
    ).load()


with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={"tracking": {"experiment_name": EXPERIMENT, "run_name": "initial-sweep"}},
) as training_session:
    training_session.run(pipeline_name="training")
    initial_kedro_version = training_session.session_id

initial_bundle = load_bundle(initial_kedro_version)
initial_report = initial_bundle["report"]
display(pd.DataFrame(initial_report["trials"]).sort_values("rmse"))
display(
    pd.DataFrame(
        {
            "Lasso": initial_report["test_metrics"],
            "Baseline": initial_report["baseline_test_metrics"],
        }
    )
)
print("Kedro version:", initial_kedro_version)
print("MLflow run:", initial_report["run_id"])
print("Model URI:", initial_report["model_uri"])

### Inspect the experiment in the UI

Run the command printed in section 0 in a terminal. Open http://127.0.0.1:5001 in a
new browser tab. Select **Model training**, **Experiments**, and `workshop-lifecycle`.
Under **Runs**, expand the arrow beside `initial-sweep`. Each `alpha=...` has
`validation_rmse`; test metrics belong to the parent run. Open its details to inspect
metrics, parameters, and the `configuration.json`, `split_ids.json`, and report artifacts.
Detailed tab labels may depend on window width; widen the window if they are hidden.

**Exercise C:** identify the winning `alpha` by validation RMSE and show that child runs
do not contain `test_rmse`. Do not select `alpha` using test results.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_registry_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri, registry_uri=tracking_uri)
experiment_id = client.get_experiment_by_name(EXPERIMENT).experiment_id
children = client.search_runs(
    [experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{initial_report['run_id']}'",
)
assert len(children) == len(context.params["model"]["alphas"])
assert all("test_rmse" not in run.data.metrics for run in children)
display(pd.DataFrame([{"alpha": run.data.params["alpha"], **run.data.metrics} for run in children]))

trials = pd.DataFrame(initial_report["trials"]).sort_values("alpha")
trials.plot(x="alpha", y="rmse", marker="o", logx=True, figsize=(8, 4))
plt.ylabel("Validation RMSE [PLN]")
plt.show()

## 4. Quality gate, registry, and release manifest

A successful run is not sufficient for deployment. Check the previously agreed thresholds.
First demonstrate rejection with a modified report, without training a bad model.
The MLflow registry stores versions; `champion` is a mutable alias.
A manifest records concrete MLflow and Kedro versions plus the bundle checksum.
Changing an alias does not automatically switch the Kedro catalog.

In [ ]:
from copy import deepcopy

initial_gate = quality_gate(initial_report, policy)
display(initial_gate)
bad_report = deepcopy(initial_report)
bad_report["test_metrics"]["rmse"] = 200_000
assert not quality_gate(bad_report, policy)["passed"]
assert initial_gate["passed"], "Stop here: this candidate must not be released."

initial_registered = mlflow.register_model(initial_report["model_uri"], REGISTERED_NAME)
assert (
    client.get_model_version(REGISTERED_NAME, initial_registered.version).run_id
    == initial_report["run_id"]
)
client.set_model_version_tag(
    REGISTERED_NAME, initial_registered.version, "kedro_bundle_version", initial_kedro_version
)
client.set_model_version_tag(REGISTERED_NAME, initial_registered.version, "quality_gate", "passed")
client.set_registered_model_alias(REGISTERED_NAME, "champion", initial_registered.version)
print("Registry version:", initial_registered.version)
print("Alias:", client.get_model_version_by_alias(REGISTERED_NAME, "champion").version)

### Record the release decision

`releases/initial.json` links code commit → lockfile → MLflow run → registry version →
Kedro bundle version. Store it as a run artifact too. This manifest records a local
teaching decision; it does not replace team access control and approval processes.
Do not edit it to bypass the gate.

In [ ]:
import hashlib
import json
import subprocess
from datetime import UTC, datetime


def write_release(bundle, kedro_version, registry_version, gate, label):
    if not gate["passed"]:
        raise ValueError("Refusing to create a release for a rejected model.")
    report = bundle["report"]
    bundle_file = WORKSPACE / "models" / "lasso.pkl" / kedro_version / "lasso.pkl"
    commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True
    ).stdout.strip()
    manifest = {
        "created_at": datetime.now(UTC).isoformat(),
        "data_dir": WORKSPACE.as_posix(),
        "tracking_uri": tracking_uri,
        "run_id": report["run_id"],
        "model_uri": report["model_uri"],
        "registered_name": REGISTERED_NAME,
        "registry_version": str(registry_version),
        "kedro_bundle_version": kedro_version,
        "bundle_sha256": hashlib.sha256(bundle_file.read_bytes()).hexdigest(),
        "code_commit": commit,
        "uv_lock_sha256": hashlib.sha256((PROJECT_ROOT / "uv.lock").read_bytes()).hexdigest(),
        "data_fingerprints": report["fingerprints"],
        "quality_gate": gate,
    }
    directory = WORKSPACE / "releases"
    directory.mkdir(exist_ok=True)
    path = directory / f"{label}.json"
    path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    client.log_artifact(report["run_id"], str(path), artifact_path="releases")
    return path


initial_release = write_release(
    initial_bundle, initial_kedro_version, initial_registered.version, initial_gate, "initial"
)
print("Initial release:", initial_release)
print(initial_release.read_text(encoding="utf-8"))

## 5. Deploy as a batch outside the notebook

This cell starts a separate Python process. The same script can run from a terminal or
an external scheduler. It checks manifest, registry, and bundle lineage, then calls
`session.run(pipeline_name="scoring", load_versions={"model_bundle": version})`.
It does not select the latest model.

The warehouse receives 250 predictions in one transaction. The number of training runs
must stay unchanged. Coloured Kedro logs in the terminal are expected.

In [ ]:
def run_release(path):
    command = [sys.executable, "-m", "trening.score_release", "--release", str(path)]
    print("Terminal command:", f'uv run python -m trening.score_release --release "{path}"')
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    print(result.stdout[-1400:])


runs_before = len(client.search_runs([experiment_id]))
run_release(initial_release)
assert len(client.search_runs([experiment_id])) == runs_before

### Inspect the second database and verify reproducibility

Link predictions to a `run_id`, not just a model name. The query shows batch history.
Also load a concrete registry version without Kedro and compare predictions.
**Exercise D:** run the batch again in a terminal. Batch count grows while training-run
count stays unchanged. Rerunning the pipeline creates a new batch ID; it is not idempotent.

In [ ]:
from sqlalchemy import create_engine, text


def warehouse_query(sql):
    engine = create_engine(config["credentials"]["warehouse_db"]["con"])
    try:
        with engine.connect() as connection:
            return pd.read_sql(text(sql), connection)
    finally:
        engine.dispose()


history_sql = """SELECT b.batch_id, b.run_id, b.n_rows, b.created_at, m.best_alpha
FROM prediction_batches b JOIN model_runs m ON b.run_id = m.run_id
ORDER BY b.created_at, b.batch_id"""
display(warehouse_query(history_sql))
display(warehouse_query("SELECT * FROM predictions LIMIT 5"))
assert warehouse_query(history_sql).iloc[-1]["n_rows"] == 250

restored = mlflow.sklearn.load_model(f"models:/{REGISTERED_NAME}/{initial_registered.version}")
np.testing.assert_allclose(
    restored.predict(raw_scoring[features]), initial_bundle["model"].predict(raw_scoring[features])
)

## 6. Monitor inputs, and measure quality only when labels arrive

First shift area by 50 m² **only in an in-memory copy**. A teaching alert means a mean
shift above 0.5 reference standard deviations or a missing-rate change above 0.10.
This is a simple heuristic, not a statistical test. Without actual prices, RMSE cannot
be measured and input change alone does not prove model degradation.

In [ ]:
shifted_inputs = raw_scoring.copy()
shifted_inputs["area_m2"] += 50
validate_frame(shifted_inputs, schema, training=False)
input_monitor = monitor_inputs(clean, shifted_inputs, features)
display(input_monitor)
assert bool(input_monitor.loc["area_m2", "alert"])

### Prices from a new period have arrived

Reuse the project's generator with a different seed and IDs. Add PLN 120,000 to the target:
this simulates a price-level change that the feature distribution alone cannot reveal.
These are independent monitoring records; do not fit a model on them. Record monitoring
in a separate MLflow experiment linked to the deployed model's run.
**Exercise E:** compare an input alert with a quality alert. Which needs labels?

In [ ]:
monitoring = future_frame(WORKSPACE, "monitoring", seed=84, id_offset=100_000)
monitoring = validate_frame(monitoring, schema, training=True)
initial_future_metrics = regression_metrics(
    monitoring[schema["target"]], restored.predict(monitoring[features])
)
display(initial_future_metrics)
assert initial_future_metrics["rmse"] > policy["max_rmse_pln"]

monitor_experiment = mlflow.set_experiment("workshop-monitoring")
with mlflow.start_run(
    experiment_id=monitor_experiment.experiment_id, run_name="labelled-future-period"
):
    mlflow.set_tags(
        {"model_run_id": initial_report["run_id"], "registry_version": initial_registered.version}
    )
    mlflow.log_metrics({f"observed_{key}": value for key, value in initial_future_metrics.items()})
    mlflow.log_dict(
        input_monitor.reset_index(names="feature").to_dict(orient="records"), "input_monitor.json"
    )
    mlflow.log_param("quality_alert", True)

## 7. Retrain on the new period

Use **different** records from monitoring: another seed and disjoint IDs. Replace training
and scoring tables only inside this exercise's database. Save source v1 as Parquet.
The pipeline and model configuration stay the same; change the run and dataset labels.
Neither monitoring data nor the final new holdout participates in fitting.

In [ ]:
future_training = future_frame(WORKSPACE, "retraining", seed=73, id_offset=10_000, rows=1500)
future_scoring = future_frame(WORKSPACE, "scoring", seed=97, id_offset=30_000, rows=250).drop(
    columns=[schema["target"]]
)
assert not set(future_training.property_id) & set(monitoring.property_id)
raw.to_parquet(WORKSPACE / "source" / "training_v1.parquet", index=False)
raw_scoring.to_parquet(WORKSPACE / "source" / "scoring_v1.parquet", index=False)
source_engine = create_engine(config["credentials"]["source_db"]["con"])
try:
    with source_engine.begin() as connection:
        future_training.to_sql("training_properties", connection, if_exists="replace", index=False)
        future_scoring.to_sql("scoring_properties", connection, if_exists="replace", index=False)
finally:
    source_engine.dispose()

with KedroSession.create(
    project_path=PROJECT_ROOT,
    runtime_params={
        "tracking": {
            "experiment_name": EXPERIMENT,
            "run_name": "retrained-sweep",
            "dataset_name": "synthetic-future-v2",
        }
    },
) as retraining_session:
    retraining_session.run(pipeline_name="training")
    retrained_kedro_version = retraining_session.session_id

retrained_bundle = load_bundle(retrained_kedro_version)
retrained_report = retrained_bundle["report"]
assert retrained_report["run_id"] != initial_report["run_id"]
assert retrained_report["fingerprints"] != initial_report["fingerprints"]
display(
    pd.DataFrame(
        {
            "Initial test": initial_report["test_metrics"],
            "New-period test": retrained_report["test_metrics"],
        }
    )
)

### Challenger versus champion on a fresh independent holdout

Compare both models on **the same new records**, with another seed and disjoint IDs.
The previous table's test metrics do not compare performance on the new period.
Keep the original quality budget and additionally require at least a 50% RMSE reduction
versus the previous champion on this holdout. This is one teaching decision; further
tuning would require another final dataset.

In [ ]:
holdout = future_frame(WORKSPACE, "acceptance", seed=91, id_offset=20_000)
assert not set(holdout.property_id) & (
    set(future_training.property_id) | set(monitoring.property_id)
)
old_metrics = regression_metrics(
    holdout[schema["target"]], initial_bundle["model"].predict(holdout[features])
)
new_metrics = regression_metrics(
    holdout[schema["target"]], retrained_bundle["model"].predict(holdout[features])
)
display(pd.DataFrame({"Old champion": old_metrics, "Challenger": new_metrics}))
retrained_gate = quality_gate(retrained_report, policy)
independent_passed = (
    new_metrics["rmse"] <= policy["max_rmse_pln"]
    and new_metrics["rmse"] < old_metrics["rmse"] * 0.5
)
retrained_gate["checks"]["independent_period_passed"] = bool(independent_passed)
retrained_gate["independent_period"] = {
    "seed": 91,
    "old_metrics": old_metrics,
    "new_metrics": new_metrics,
}
retrained_gate["passed"] = bool(retrained_gate["passed"] and independent_passed)
assert retrained_gate["passed"], "Keep the current release; investigate this candidate."

retrained_registered = mlflow.register_model(retrained_report["model_uri"], REGISTERED_NAME)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "kedro_bundle_version", retrained_kedro_version
)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "quality_gate", "passed"
)
client.set_registered_model_alias(REGISTERED_NAME, "candidate", retrained_registered.version)
retrained_release = write_release(
    retrained_bundle,
    retrained_kedro_version,
    retrained_registered.version,
    retrained_gate,
    "retrained",
)
client.set_registered_model_alias(REGISTERED_NAME, "champion", retrained_registered.version)
client.set_registered_model_alias(REGISTERED_NAME, "previous", initial_registered.version)
run_release(retrained_release)
assert warehouse_query(history_sql).iloc[-1]["run_id"] == retrained_report["run_id"]

## 8. Rollback: restore a concrete version

Simulate a deployment incident: switch the alias and execute the **old manifest**.
The newest saved bundle is still v2, but the batch must use v1. Prediction history remains.
In this synthetic new regime, v1 has worse quality; rollback here is an operational drill.
Return to the accepted v2 after checking it.
**Exercise F:** explain why changing only the alias cannot switch a job with a pinned
manifest, and why a rollback can reduce business quality.

In [ ]:
client.set_registered_model_alias(REGISTERED_NAME, "champion", initial_registered.version)
runs_before_rollback = len(client.search_runs([experiment_id]))
run_release(initial_release)
rollback_batch = warehouse_query(history_sql).iloc[-1]
assert rollback_batch["run_id"] == initial_report["run_id"]
assert len(client.search_runs([experiment_id])) == runs_before_rollback
rollback_predictions = warehouse_query(
    "SELECT property_id, prediction FROM predictions "
    f"WHERE batch_id = '{rollback_batch['batch_id']}' ORDER BY property_id"
)
np.testing.assert_allclose(
    rollback_predictions.prediction,
    initial_bundle["model"].predict(future_scoring.sort_values("property_id")[features]),
)
print("Rollback verified:", rollback_batch["run_id"])
display(warehouse_query(history_sql))

## 9. Finish the exercise and retain the evidence

Restore the accepted v2. Keep v1 available for rollback; do not delete history or artifacts.
Agree on retention and model retirement policies separately. `summary.json` records the
exercise result. Manifests and reports are in `WORKSPACE`.

**Final check:** identify the model run used by the latest batch, explain alias versus
version, and run scoring without training.
In JupyterLab select **File → Save Notebook**. In the MLflow terminal press **Ctrl+C**.
Before running everything again select **Kernel → Restart Kernel and Clear Outputs of
All Cells…**. Do not delete databases to restart: a fresh notebook execution creates a
new directory. See the instructor guide and final assignment too.

In [ ]:
client.set_registered_model_alias(REGISTERED_NAME, "champion", retrained_registered.version)
run_release(retrained_release)
client.set_model_version_tag(
    REGISTERED_NAME, initial_registered.version, "deployment_status", "rollback_available"
)
client.set_model_version_tag(
    REGISTERED_NAME, retrained_registered.version, "deployment_status", "active"
)
history = warehouse_query(history_sql)
assert history.iloc[-1]["run_id"] == retrained_report["run_id"]
assert str(client.get_model_version_by_alias(REGISTERED_NAME, "champion").version) == str(
    retrained_registered.version
)
assert history.n_rows.eq(250).all()
assert set(history.run_id) == {initial_report["run_id"], retrained_report["run_id"]}
summary = {
    "language": LANGUAGE,
    "workspace": str(WORKSPACE),
    "initial_run_id": initial_report["run_id"],
    "active_run_id": retrained_report["run_id"],
    "batch_count": len(history),
    "rollback_verified": True,
    "retraining_improved_rmse": bool(independent_passed),
    "initial_release": str(initial_release),
    "active_release": str(retrained_release),
}
(WORKSPACE / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(history)
display(summary)

## 10. Where the demonstration ends

Completed: goal and criteria → data and contracts → EDA → validation → experiments →
artifact → registry → release decision → batch into a separate database → monitoring →
retraining → rollback → retained evidence. Monitoring and promotion are triggered manually.
A shared deployment would need separate access control, PostgreSQL, object storage,
migrations, retention, orchestration, real labels, and time-aware validation.
IDs and seeds establish separation in this exercise, not real temporal market independence.
Do not use the synthetic generator to estimate real business performance.

Sources: [Kedro and Jupyter](https://docs.kedro.org/en/stable/integrations-and-plugins/notebooks_and_ipython/kedro_and_notebooks/),
[MLflow Model Registry](https://mlflow.org/docs/latest/ml/model-registry/workflow/).
Training lives in `src/kedro_lasso/training.py`; teaching utilities live in `trening/`.